# RSNA Knee — baseline (handcrafted features + LightGBM)
Kaggle Notebook用。リポジトリの `src/` を使う。提出時(オフライン)は `src/` をDatasetとして追加し、`SRC` を書き換える。

In [ ]:
import os, sys, subprocess, warnings
os.environ['PYTHONWARNINGS'] = 'ignore'  # joblibのワーカープロセスにも効かせる
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from pathlib import Path
from joblib import Parallel, delayed
_name = 'rsna-knee-abnormality-detection'
_cands = [Path('/kaggle/input/competitions')/_name, Path('/kaggle/input')/_name]
COMP = next((d for d in _cands if (d/'train.csv').exists()), None)
assert COMP, f"train.csv が見つからない。/kaggle/input: {[p.name for p in Path('/kaggle/input').iterdir()]}"
print('COMP =', COMP)
SRC = Path('/kaggle/working/repo/rsna-knee-abnormality-detection/src')
if not SRC.exists():  # internet ON のときだけ。OFFなら src を Dataset で追加して SRC を変更
    subprocess.run(['git','clone','--depth','1','-b','main',
                    'https://github.com/hidenori24/kaggle_ipynb','/kaggle/working/repo'], check=True)
sys.path.insert(0, str(SRC))
from pseudo_labels import LABELS, add_pseudo_labels
from features import study_features


In [ ]:
train = pd.read_csv(COMP/'train.csv'); tseries = pd.read_csv(COMP/'train_series.csv')
test = pd.read_csv(COMP/'test.csv');   teseries = pd.read_csv(COMP/'test_series.csv')
print(train.shape, tseries.shape, test.shape)
print(train[LABELS].notna().sum())   # 正解ラベルがあるstudy数
train['has_true'] = train['ACL'].notna()   # 正解ラベルあり(疑似ラベルではない)
train = add_pseudo_labels(train)
print(train[LABELS].mean().round(3))

In [ ]:
def feats(df, series, root, n_jobs=4):
    g = dict(tuple(series.groupby('StudyInstanceUID')))
    def f(uid):
        s = g.get(uid)
        return study_features(root/uid, s) if s is not None else None
    out = Parallel(n_jobs=n_jobs)(delayed(f)(u) for u in df['StudyInstanceUID'])
    dim = next(o for o in out if o is not None).shape[0]
    return np.stack([o if o is not None else np.zeros(dim, np.float32) for o in out])
N = 600  # まず少数で動作確認。慣れたら len(train) に
rest = train[~train.has_true]
tr = pd.concat([train[train.has_true], rest.sample(min(N, len(rest)), random_state=0)]).reset_index(drop=True)
Xtr = feats(tr, tseries, COMP/'train_series'); print(Xtr.shape)

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
oof = np.zeros((len(tr), len(LABELS))); 
kf = KFold(5, shuffle=True, random_state=0)
params = dict(n_estimators=200, learning_rate=0.03, num_leaves=8, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.3, verbose=-1)
for j, c in enumerate(LABELS):
    y = tr[c].values.astype(int)
    if y.min() == y.max(): oof[:, j] = 0.5; continue
    for a, b in kf.split(Xtr):
        m = lgb.LGBMClassifier(**params).fit(Xtr[a], y[a]); oof[b, j] = m.predict_proba(Xtr[b])[:, 1]
aucs = {c: roc_auc_score(tr[c], oof[:, j]) for j, c in enumerate(LABELS) if tr[c].nunique() > 1}
print(pd.Series(aucs).round(3)); print('macro AUC (vs pseudo-labels):', np.mean(list(aucs.values())))
# 正解ラベルのみでの評価(件数が少なくブレる。傾向を見る用)
m = tr.has_true.values
t_aucs = {c: roc_auc_score(tr.loc[m, c], oof[m, j]) for j, c in enumerate(LABELS) if tr.loc[m, c].nunique() > 1}
print('macro AUC (vs TRUE labels, n=%d):' % m.sum(), np.mean(list(t_aucs.values())))

In [ ]:
Xte = feats(test, teseries, COMP/'test_series')
sub = pd.DataFrame({'StudyInstanceUID': test['StudyInstanceUID']})
for j, c in enumerate(LABELS):
    y = tr[c].values.astype(int)
    sub[c] = 0.5 if y.min() == y.max() else lgb.LGBMClassifier(**params).fit(Xtr, y).predict_proba(Xte)[:, 1]
sub.to_csv('submission.csv', index=False); sub.head()